# Test 3: Ablation Study â€” Does the DFG Actually Help?

**Goal**: Isolate the contribution of the **Data Flow Graph (DFG)** by comparing:

| Model | Backbone | DFG Attention? |
|---|---|---|
| A â€” `GraphCodeBERT + DFG` | GraphCodeBERT | âœ… Full DFG-aware sparse mask |
| B â€” `GraphCodeBERT (no DFG)` | GraphCodeBERT | âŒ Standard causal attention, text-only |

Both models share the **same backbone weights and architecture** â€” only the
attention mask and position representation differ between runs.
Any performance gap is directly attributable to the DFG.

**Kaggle inputs**:
- `/kaggle/input/dfgdataset2/dataset_graphcodebert.jsonl`
- `/kaggle/input/model2/model.bin` â€” pre-trained GraphCodeBERT+DFG weights

**Output**: `/kaggle/working/test3_ablation_results.txt`

In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 25.9 MB/s eta 0:00:00


In [2]:
import os, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    get_linear_schedule_with_warmup,
    RobertaConfig, RobertaModel,
    AutoTokenizer)

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

print("Imports OK")
print(f"CUDA: {torch.cuda.is_available()}")

Imports OK
CUDA: True


In [3]:
# â”€â”€â”€ CONFIGURATION â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class Args:
    train_file          = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    gcb_weights         = "/kaggle/input/notebooks/hasanmahmudabdullah2/graphcodebert-v2-training/saved_models/model.bin"  # trained GraphCodeBERT+DFG
    output_dir_nodeg    = "saved_models_ablation_nodg"       # trained no-DFG model

    model_name_or_path  = "microsoft/graphcodebert-base"
    tokenizer_name      = "microsoft/graphcodebert-base"

    code_length         = 384
    data_flow_length    = 128
    train_batch_size    = 16
    eval_batch_size     = 32
    learning_rate       = 2e-5
    max_grad_norm       = 1.0
    num_train_epochs    = 3
    seed                = 42
    val_ratio           = 0.10    # 90/10 split

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if args.n_gpu > 0:
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)
print(f"Device: {args.device}")

Device: cuda


In [4]:
# â”€â”€â”€ SHARED CLASSIFICATION HEAD (same for both conditions) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class ClassificationModel(nn.Module):
    """Generic head wrapping a RobertaModel encoder."""
    def __init__(self, encoder, config):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def _classify(self, cls_repr, labels):
        logits = self.classifier(self.dropout(cls_repr))
        prob   = F.softmax(logits, dim=-1)
        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob


# â”€â”€â”€ Condition A: full DFG-aware attention â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class ModelWithDFG(ClassificationModel):
    def forward(self, input_ids=None, p_ids=None, attn_mask=None, labels=None):
        ext = (1.0 - attn_mask) * -10000.0
        ext = ext.unsqueeze(1)
        emb = self.encoder.embeddings(input_ids=input_ids, position_ids=p_ids)
        out = self.encoder.encoder(
            emb, attention_mask=ext,
            head_mask=[None] * self.config.num_hidden_layers
        )[0]
        return self._classify(out[:, 0, :], labels)


# â”€â”€â”€ Condition B: same backbone, standard text attention (no DFG) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class ModelNoDFG(ClassificationModel):
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        # Standard forward â€” GraphCodeBERT backbone, but just use it like RoBERTa
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        return self._classify(out[:, 0, :], labels)

In [5]:
# â”€â”€â”€ DATASET A â€” with DFG (full GraphCodeBERT format) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class TextDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args      = args
        self.tokenizer = tokenizer
        self.total_len = args.code_length + args.data_flow_length
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def _char_idx(self, lines, coord):
        r, c = coord
        return sum(len(lines[i]) for i in range(min(r, len(lines)))) + c

    def __getitem__(self, idx):
        entry      = json.loads(self.lines[idx])
        code       = entry.get('code', '')
        dfg        = entry.get('dfg', [])[:self.args.data_flow_length]
        label      = int(entry.get('label', 0))
        code_lines = code.splitlines(keepends=True)

        tok = self.tokenizer(
            code, max_length=self.args.code_length, truncation=True,
            padding='max_length', return_offsets_mapping=True
        )
        input_ids, offsets = tok['input_ids'], tok['offset_mapping']
        dfg_ids = [self.tokenizer.unk_token_id] * len(dfg)
        n2t, p2n = {}, {}

        for ni, node in enumerate(dfg):
            sp, ep = node[1][0], node[1][1]
            pk = (sp[0], sp[1], ep[0], ep[1])
            p2n[pk] = ni
            try:
                cs = self._char_idx(code_lines, sp)
                ce = self._char_idx(code_lines, ep)
                n2t[ni] = [i for i, (ts, te) in enumerate(offsets)
                           if ts != te and
                           ((ts >= cs and te <= ce) or (cs >= ts and ce <= te))]
            except IndexError:
                n2t[ni] = []

        c_len = self.args.code_length
        mask  = np.zeros((self.total_len, self.total_len), dtype=bool)
        mask[:c_len, :c_len] = True
        for ni, node in enumerate(dfg):
            ani = c_len + ni
            for ti in n2t.get(ni, []):
                mask[ani, ti] = mask[ti, ani] = True
            for pp in node[4]:
                pk = (pp[0][0], pp[0][1], pp[1][0], pp[1][1])
                if pk in p2n:
                    mask[ani, c_len + p2n[pk]] = mask[c_len + p2n[pk], ani] = True
            mask[ani, ani] = True

        ids   = input_ids + dfg_ids
        p_ids = [i + 2 for i in range(c_len)] + [0] * len(dfg_ids)
        pad   = self.total_len - len(ids)
        if pad > 0:
            ids   += [self.tokenizer.pad_token_id] * pad
            p_ids += [1] * pad

        return {
            'input_ids': torch.tensor(ids,          dtype=torch.long),
            'p_ids':     torch.tensor(p_ids,        dtype=torch.long),
            'attn_mask': torch.tensor(mask,         dtype=torch.float),
            'label':     torch.tensor(label,        dtype=torch.long)
        }


# â”€â”€â”€ DATASET B â€” code only (no DFG tokens appended) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class TextDatasetNoDFG(Dataset):
    """Same underlying file, but strips DFG â€” uses GraphCodeBERT tokenizer."""
    def __init__(self, tokenizer, args, file_path):
        self.tokenizer = tokenizer
        self.args      = args
        with open(file_path) as f:
            self.lines = f.readlines()

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        try:
            entry = json.loads(self.lines[idx])
        except Exception:
            return self.__getitem__((idx + 1) % len(self.lines))
        code  = entry.get('code', '')
        label = int(entry.get('label', 0))
        tok   = self.tokenizer(
            code,
            max_length=self.args.code_length,  # code_length only â€” no DFG slots
            truncation=True,
            padding='max_length'
        )
        return {
            'input_ids':      torch.tensor(tok['input_ids'],      dtype=torch.long),
            'attention_mask': torch.tensor(tok['attention_mask'], dtype=torch.long),
            'label':          torch.tensor(label,                 dtype=torch.long)
        }

In [6]:
# Evaluation helpers
@torch.no_grad()
def eval_with_dfg(model, dataset, desc='Eval (DFG)'):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        num_workers=2, pin_memory=True)
    model.eval()
    preds, labels, probs_list = [], [], []
    for batch in tqdm(loader, desc=desc):
        inp = {
            'input_ids': batch['input_ids'].to(args.device),
            'p_ids':     batch['p_ids'].to(args.device),
            'attn_mask': batch['attn_mask'].to(args.device)
        }
        prob = model(**inp)
        probs_list.append(prob.cpu().numpy())
        preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    model.train()
    return np.array(preds), np.array(labels), np.concatenate(probs_list, axis=0)


@torch.no_grad()
def eval_no_dfg(model, dataset, desc='Eval (no DFG)'):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size,
                        num_workers=2, pin_memory=True)
    model.eval()
    preds, labels, probs_list = [], [], []
    for batch in tqdm(loader, desc=desc):
        inp = {
            'input_ids':      batch['input_ids'].to(args.device),
            'attention_mask': batch['attention_mask'].to(args.device)
        }
        prob = model(**inp)
        probs_list.append(prob.cpu().numpy())
        preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    model.train()
    return np.array(preds), np.array(labels), np.concatenate(probs_list, axis=0)


def report(name, preds, labels, probs):
    acc     = accuracy_score(labels, preds)
    roc_auc = roc_auc_score(labels, probs[:, 1])
    pr_auc  = average_precision_score(labels, probs[:, 1])
    cm      = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4%}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  PR-AUC   : {pr_auc:.4f}")
    print(f"  FN (missed malware): {fn}")
    print()
    print(classification_report(
        labels, preds, target_names=['Safe', 'Malicious'], digits=4
    ))
    return {'name': name, 'acc': acc, 'roc_auc': roc_auc,
            'pr_auc': pr_auc, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp}

In [7]:
# â”€â”€â”€ TRAIN NO-DFG MODEL (Condition B) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Train on the full 90% training split â€” no validation during training.

def train_no_dfg(model, train_ds):
    loader    = DataLoader(train_ds, batch_size=args.train_batch_size,
                           shuffle=True, num_workers=2, pin_memory=True)
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0,
        num_training_steps=len(loader) * args.num_train_epochs
    )
    scaler = GradScaler()

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        for batch in tqdm(loader, desc=f"NoDFG Epoch {epoch}"):
            inp = {
                'input_ids':      batch['input_ids'].to(args.device),
                'attention_mask': batch['attention_mask'].to(args.device),
                'labels':         batch['label'].to(args.device)
            }
            optimizer.zero_grad()
            with autocast():
                loss, _ = model(**inp)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()
        print(f"Epoch {epoch} | Loss: {tr_loss / len(loader):.4f}")

    # Save final model after all epochs
    os.makedirs(args.output_dir_nodeg, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(args.output_dir_nodeg, "model.bin"))
    print(f"No-DFG training done. Model saved.")


In [8]:
# â”€â”€â”€ MAIN â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
set_seed(args.seed)

# â”€â”€ 1. Shared tokenizer (both conditions use GraphCodeBERT tokenizer) â”€â”€
config    = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)

# â”€â”€ 2. Condition A â€” Load trained GraphCodeBERT+DFG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("[Condition A] Loading GraphCodeBERT + DFG model...")
enc_a   = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model_a = ModelWithDFG(enc_a, config).to(args.device)
model_a.load_state_dict(torch.load(args.gcb_weights, map_location=args.device))
model_a.eval()
print("  âœ“ Loaded")

# â”€â”€ 3. Build datasets â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("\nBuilding datasets...")
ds_dfg    = TextDataset(tokenizer, args, args.train_file)
ds_no_dfg = TextDatasetNoDFG(tokenizer, args, args.train_file)

n        = len(ds_dfg)
train_n  = int(0.9 * n)
test_n   = n - train_n

# Use the SAME seed so both splits select identical indices
gen_a = torch.Generator().manual_seed(args.seed)
train_ds_dfg, test_ds_dfg = random_split(ds_dfg, [train_n, test_n], generator=gen_a)

gen_b = torch.Generator().manual_seed(args.seed)
train_ds_no_dfg, test_ds_no_dfg = random_split(ds_no_dfg, [train_n, test_n], generator=gen_b)

print(f"  Total  : {n:,}")
print(f"  Train  : {train_n:,} (90%)")
print(f"  Test   : {test_n:,} (10%)")

# â”€â”€ 4. Condition B â€” Train no-DFG model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
no_dfg_ckpt = os.path.join(args.output_dir_nodeg, "model.bin")
print("\n[Condition B] Training GraphCodeBERT (no DFG)...")
enc_b   = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model_b = ModelNoDFG(enc_b, config).to(args.device)

if os.path.exists(no_dfg_ckpt):
    print(f"  Found existing checkpoint at {no_dfg_ckpt}. Loading...")
    model_b.load_state_dict(torch.load(no_dfg_ckpt, map_location=args.device))
else:
    train_no_dfg(model_b, train_ds_no_dfg)
    model_b.load_state_dict(torch.load(no_dfg_ckpt, map_location=args.device))

model_b.eval()
print("  âœ“ No-DFG model ready")


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Epoch 0 | Loss: 0.3075


NoDFG Epoch 1: 100%|██████████| 11248/11248 [1:50:10<00:00,  1.70it/s]


Epoch 1 | Loss: 0.2498


NoDFG Epoch 2: 100%|██████████| 11248/11248 [1:49:58<00:00,  1.70it/s]


Epoch 2 | Loss: 0.2094
No-DFG training done. Model saved.
  âœ“ No-DFG model ready


In [9]:
# Evaluate both conditions
print('\nEvaluating Condition A (GraphCodeBERT + DFG)...')
preds_a, labels_a, probs_a = eval_with_dfg(model_a, test_ds_dfg)

print('Evaluating Condition B (GraphCodeBERT, no DFG)...')
preds_b, labels_b, probs_b = eval_no_dfg(model_b, test_ds_no_dfg)

np.save('/kaggle/working/test_probs_dfg.npy', probs_a)
np.save('/kaggle/working/test_probs_nodfg.npy', probs_b)
np.save('/kaggle/working/test_labels.npy', labels_a)
print('Saved -> /kaggle/working/test_probs_dfg.npy')
print('Saved -> /kaggle/working/test_probs_nodfg.npy')
print('Saved -> /kaggle/working/test_labels.npy')

res_a = report('Condition A: GraphCodeBERT + DFG', preds_a, labels_a, probs_a)
res_b = report('Condition B: GraphCodeBERT (no DFG)', preds_b, labels_b, probs_b)


Evaluating Condition A (GraphCodeBERT + DFG)...


Eval (DFG): 100%|██████████| 625/625 [05:10<00:00,  2.01it/s]


Evaluating Condition B (GraphCodeBERT, no DFG)...


Eval (no DFG): 100%|██████████| 625/625 [03:45<00:00,  2.77it/s]

Saved -> /kaggle/working/test_probs_dfg.npy
Saved -> /kaggle/working/test_probs_nodfg.npy
Saved -> /kaggle/working/test_labels.npy

  Condition A: GraphCodeBERT + DFG
  Accuracy : 88.7077%
  ROC-AUC  : 0.9616
  PR-AUC   : 0.9622
  FN (missed malware): 1184

              precision    recall  f1-score   support

        Safe     0.8840    0.8936    0.8888     10095
   Malicious     0.8903    0.8804    0.8853      9901

    accuracy                         0.8871     19996
   macro avg     0.8871    0.8870    0.8871     19996
weighted avg     0.8871    0.8871    0.8871     19996


  Condition B: GraphCodeBERT (no DFG)
  Accuracy : 88.7177%
  ROC-AUC  : 0.9612
  PR-AUC   : 0.9618
  FN (missed malware): 1194

              precision    recall  f1-score   support

        Safe     0.8833    0.8948    0.8890     10095
   Malicious     0.8913    0.8794    0.8853      9901

    accuracy                         0.8872     19996
   macro avg     0.8873    0.8871    0.8871     19996
weighted avg 

In [10]:
# â”€â”€â”€ ABLATION DELTA SUMMARY â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("\n" + "="*55)
print("ABLATION DELTA:  A (DFG)  vs  B (no DFG)")
print("="*55)

metrics = ['acc', 'roc_auc', 'pr_auc']
labels_str = ['Accuracy', 'ROC-AUC', 'PR-AUC']
for m, lbl in zip(metrics, labels_str):
    delta = res_a[m] - res_b[m]
    arrow = 'â–²' if delta > 0 else 'â–¼'
    print(f"  {lbl:10s}: A={res_a[m]:.4f}  B={res_b[m]:.4f}  "
          f"Î”={delta:+.4f} {arrow}")

fn_delta = res_a['fn'] - res_b['fn']
print(f"\n  False Negatives (missed malware):")
print(f"    A (DFG):    {res_a['fn']}")
print(f"    B (no DFG): {res_b['fn']}")
print(f"    Î” (A - B):  {fn_delta:+d}")

if res_a['acc'] > res_b['acc']:
    print("\nâœ“ DFG attention HELPS: Condition A outperforms Condition B.")
else:
    print("\nâœ— DFG attention does NOT help, or hurts performance.")


ABLATION DELTA:  A (DFG)  vs  B (no DFG)
  Accuracy  : A=0.8871  B=0.8872  Î”=-0.0001 â–¼
  ROC-AUC   : A=0.9616  B=0.9612  Î”=+0.0004 â–²
  PR-AUC    : A=0.9622  B=0.9618  Î”=+0.0004 â–²

  False Negatives (missed malware):
    A (DFG):    1184
    B (no DFG): 1194
    Î” (A - B):  -10

âœ— DFG attention does NOT help, or hurts performance.


In [11]:
# â”€â”€â”€ BAR CHART COMPARISON â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
conditions = ['GraphCodeBERT\n+ DFG', 'GraphCodeBERT\n(no DFG)']
colors     = ['#4C72B0', '#DD8452']

metric_info = [
    ('Accuracy', [res_a['acc'],     res_b['acc']]),
    ('ROC-AUC',  [res_a['roc_auc'], res_b['roc_auc']]),
    ('PR-AUC',   [res_a['pr_auc'],  res_b['pr_auc']]),
]

for ax, (title, vals) in zip(axes, metric_info):
    bars = ax.bar(conditions, vals, color=colors, width=0.5, edgecolor='white')
    ymin = max(0, min(vals) - 0.05)
    ax.set_ylim([ymin, min(1.0, max(vals) + 0.04)])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(title, fontsize=11)
    ax.tick_params(axis='x', labelsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Ablation Study: DFG vs No-DFG on GraphCodeBERT',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = "/kaggle/working/test3_ablation_bar.png"
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"Bar chart saved â†’ {plot_path}")

Bar chart saved â†’ /kaggle/working/test3_ablation_bar.png


In [12]:
# â”€â”€â”€ SAVE RESULTS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
out_path = "/kaggle/working/test3_ablation_results.txt"
with open(out_path, "w") as f:
    f.write("Test 3: DFG Ablation Study Results\n")
    f.write("=" * 55 + "\n\n")
    for res in [res_a, res_b]:
        f.write(f"{res['name']}\n")
        f.write(f"  Accuracy : {res['acc']:.4%}\n")
        f.write(f"  ROC-AUC  : {res['roc_auc']:.4f}\n")
        f.write(f"  PR-AUC   : {res['pr_auc']:.4f}\n")
        f.write(f"  TP={res['tp']}  FP={res['fp']}  "
                f"FN={res['fn']}  TN={res['tn']}\n\n")
    f.write("Delta (A - B):\n")
    for m, lbl in zip(metrics, labels_str):
        f.write(f"  {lbl}: {res_a[m] - res_b[m]:+.4f}\n")

print(f"Results saved â†’ {out_path}")
print(f"Bar chart   â†’ /kaggle/working/test3_ablation_bar.png")

Results saved â†’ /kaggle/working/test3_ablation_results.txt
Bar chart   â†’ /kaggle/working/test3_ablation_bar.png
